In [ ]:
# =============================================================================
# Experiment 11: Malicious Data Injection Detection in Databases (Decision Tree)
# Objective: Develop a decision tree classifier to detect malicious data injections within databases. [CO5]
# =============================================================================

# 1. Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# =============================================================================
# 2. Create / Load Database Query & Data Payload Dataset
# Label: 0 = Legitimate Database Query/Input, 1 = Malicious Data Injection (SQLi)
# =============================================================================

legitimate_queries = [
    "SELECT id, name, email FROM users WHERE user_id = 1042;",
    "INSERT INTO orders (user_id, product_id, quantity) VALUES (55, 102, 2);",
    "UPDATE user_profiles SET city = 'Bengaluru' WHERE user_id = 88;",
    "SELECT title, price FROM products WHERE category = 'Electronics' ORDER BY price ASC;",
    "DELETE FROM cart_items WHERE session_id = 'sess_99281a' AND item_id = 14;",
    "SELECT COUNT(*) FROM transactions WHERE status = 'COMPLETED' AND amount > 500;",
    "INSERT INTO audit_logs (event_type, created_at) VALUES ('LOGIN_SUCCESS', '2026-11-14');",
    "SELECT department, AVG(salary) FROM employees GROUP BY department;",
    "UPDATE inventory SET stock = stock - 1 WHERE sku = 'LAPTOP-009';",
    "SELECT first_name, last_name FROM students WHERE usn = '25MCAR0046';"
] * 35

malicious_injections = [
    "SELECT * FROM users WHERE username = 'admin' OR '1'='1' --';",
    "SELECT name, password FROM users WHERE id = 1 UNION SELECT credit_card, cvv FROM payments;",
    "INSERT INTO users (username) VALUES ('hacker'); DROP TABLE audit_logs; --');",
    "SELECT * FROM accounts WHERE user_id = 10 OR 1=1#;",
    "UPDATE users SET role = 'ADMIN' WHERE email = 'attacker@evil.com' /* bypass */;",
    "SELECT * FROM products WHERE id = 5; EXEC xp_cmdshell('net user attacker pass123 /add');",
    "SELECT * FROM login WHERE user = '' OR TRUE --' AND pass = '';",
    "SELECT id FROM orders WHERE id = 1 UNION ALL SELECT table_name, column_name FROM information_schema.columns;",
    "SELECT * FROM users WHERE id = 1 AND SLEEP(5) --;",
    "INSERT INTO comments (body) VALUES ('test'); TRUNCATE TABLE user_sessions; --');"
] * 35

queries = legitimate_queries + malicious_injections
labels = [0] * len(legitimate_queries) + [1] * len(malicious_injections)

df = pd.DataFrame({'SQL_Query': queries, 'Label': labels})
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Total Queries: {len(df)} | Legitimate (0): {(df['Label']==0).sum()} | Malicious Injection (1): {(df['Label']==1).sum()}")
print(df.head())

In [ ]:
# =============================================================================
# 3. Extract Structural & Syntactic SQL Injection Features
# =============================================================================

def extract_sqli_features(query):
    q_lower = query.lower()
    features = {
        'query_length': len(query),
        'single_quote_count': query.count("'"),
        'semicolon_count': query.count(';'),
        'has_comment_syntax': 1 if re.search(r'(--|#|/\*)', query) else 0,
        'has_tautology': 1 if re.search(r"(\bor\b\s+['\"]?1['\"]?\s*=\s*['\"]?1['\"]?|\bor\b\s+true)", q_lower) else 0,
        'has_union_select': 1 if re.search(r'\bunion\b.*\bselect\b', q_lower) else 0,
        'has_destructive_cmd': 1 if re.search(r'\b(drop|truncate|exec|xp_cmdshell|sleep|information_schema)\b', q_lower) else 0,
        'special_char_count': len(re.findall(r"['\"=;\-\*/#]", query))
    }
    return pd.Series(features)

X = df['SQL_Query'].apply(extract_sqli_features)
y = df['Label']

print("--- Extracted Database Injection Features ---")
print(X.head())

In [ ]:
# =============================================================================
# 4. Train Decision Tree Classifier
# =============================================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

dt_classifier = DecisionTreeClassifier(criterion='gini', max_depth=4, random_state=42)
dt_classifier.fit(X_train, y_train)
y_pred = dt_classifier.predict(X_test)

print(f"Decision Tree Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print("--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Malicious Injection']))

In [ ]:
# =============================================================================
# 5. Visualize Confusion Matrix & Decision Tree Rules
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Legitimate', 'Injection'], yticklabels=['Legitimate', 'Injection'], ax=axes[0])
axes[0].set_title('Database Injection Confusion Matrix')
axes[0].set_xlabel('Predicted Class')
axes[0].set_ylabel('Actual Class')

# Plot Decision Tree Structure
plot_tree(
    dt_classifier,
    feature_names=X.columns,
    class_names=['Legitimate', 'Injection'],
    filled=True,
    rounded=True,
    fontsize=9,
    ax=axes[1]
)
axes[1].set_title('Decision Tree Rules for Malicious Injection Detection')

plt.tight_layout()
plt.show()

# Test with custom database queries
sample_queries = [
    "SELECT email FROM users WHERE id = 204;",
    "SELECT * FROM users WHERE name = '' OR '1'='1' --';"
]
sample_feats = pd.Series(sample_queries).apply(extract_sqli_features)
sample_preds = dt_classifier.predict(sample_feats)

print("--- Live Database Query Inspection ---")
for q, p in zip(sample_queries, sample_preds):
    print(f"[{'MALICIOUS INJECTION' if p == 1 else 'SAFE QUERY'}] -> {q}")